In [9]:
import threading
import numpy as np
import time
from PIL import Image
from constants import PGM
import multiprocessing

In [ ]:
def sum_section(vector, results, start_idx, end_idx, result_lock, thread_id):
    """Sum a section of the vector and add to total"""
    # Calculate partial sum for this section
    partial_sum = np.sum(vector[start_idx:end_idx])

    # Need lock when updating shared result
    with result_lock:
        results[0] += partial_sum
    print(
        f"Sum thread {thread_id} processed indices {start_idx}-{end_idx}, partial sum: {partial_sum}"
    )

In [ ]:
def calculate_image_threads(height, min_rows_per_thread=50, max_threads=16):
    """
    Calculate threads for image processing

    Args:
        height: Image height in pixels
        min_rows_per_thread: Minimum rows to avoid overhead
        max_threads: Upper limit
    """
    cpu_cores = multiprocessing.cpu_count()

    # Maximum threads possible based on rows
    max_threads_by_rows = max(1, height // min_rows_per_thread)

    # Optimal threads (can't have more threads than rows)
    optimal = min(cpu_cores, max_threads_by_rows, max_threads)

    # Keep at least one CPU core free when all cores would be used
    if optimal == cpu_cores and optimal > 1:
        optimal -= 1

    # Ensure each thread gets reasonable work
    if optimal > 1:
        rows_per_thread = height // optimal
        if rows_per_thread < min_rows_per_thread:
            optimal = max(1, height // min_rows_per_thread)
            if optimal == cpu_cores and optimal > 1:
                optimal -= 1

    return optimal


if __name__ == "__main__":
    test_images = [
        (100, 100),      # Small
        (480, 640),      # VGA
        (1080, 1920),    # Full HD
        (2160, 3840),    # 4K
    ]
    
    for height, width in test_images:
        threads = calculate_image_threads(height)
        rows_per_thread = height // threads
        print(f"{height}x{width}: {threads} threads, ~{rows_per_thread} rows/thread")


100x100: 2 threads, ~50 rows/thread
480x640: 8 threads, ~60 rows/thread
1080x1920: 8 threads, ~135 rows/thread
2160x3840: 8 threads, ~270 rows/thread


In [15]:
def get_image_thread_results(image_path="lena.pgm"):
    with Image.open(image_path) as img:
        arr = np.array(img)
        data = arr.tobytes()
        metadata = PGM(img.height, img.width, int(arr.max()), data)

    height, width = metadata.height, metadata.width
    threads = calculate_image_threads(height)
    rows_per_thread = height // threads

    return {
        "metadata": metadata,
        "height": height,
        "width": width,
        "threads": threads,
        "rows_per_thread": rows_per_thread,
    }


if __name__ == "__main__":
    results = get_image_thread_results("lena.pgm")
    print(
        f"{results['height']}x{results['width']}: "
        f"{results['threads']} threads, ~{results['rows_per_thread']} rows/thread"
    )

512x512: 8 threads, ~64 rows/thread


In [ ]:
if __name__ == "__main__":
    # ---------- SUM EXAMPLE ----------
    print("\n--- SUM WITH 2 THREADS ---")

    # Using list for mutable shared result (or array with single element)
    sum_result = [0.0]  # Shared mutable result
    sum_lock = threading.Lock()

    # Split vector into 2 sections
    mid_point = vector_size // 2

    # Create 2 sum threads
    thread_sum1 = threading.Thread(
        target=sum_section, args=(vector, sum_result, 0, mid_point, sum_lock, 1)
    )
    thread_sum2 = threading.Thread(
        target=sum_section, args=(vector, sum_result, mid_point, vector_size, sum_lock, 2)
    )

    # Start threads
    start_time = time.time()
    thread_sum1.start()
    thread_sum2.start()

    # Wait for completion
    thread_sum1.join()
    thread_sum2.join()

    print(f"Total sum: {sum_result[0]}")
    print(f"Verification (np.sum): {np.sum(vector)}")
    print(f"Time taken: {time.time() - start_time:.4f} seconds")
